## Setting: home team win = 1, home team loss = 0

## Two baseline models: 

1. Home team always wins. 
Accuracy: 57.36%

2. Teams with better records of the last 10 games win; if same record, then home team wins.
Accuracy: 61.44%

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression 

In [2]:
import os
os.getcwd()

'c:\\kaggle'

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() 
# load the datasets
# team_stats = pd.read_csv(PROJECT_ROOT / 'NBA' / 'Stats' / 'TeamStatistics.csv')
matchups = pd.read_csv(PROJECT_ROOT / 'NBA' / 'Stats' / 'TeamStatisticsFrom2010Matchups.csv')

In [4]:
matchups

,gameId,gameDate,season,teamId_home,teamName_home,win,home_teamScore_roll10,home_fieldGoalsPercentage_roll10,home_threePointersPercentage_roll10,home_freeThrowsPercentage_roll10,...,threePointersPercentage_roll10_diff,freeThrowsPercentage_roll10_diff,reboundsOffensive_roll10_diff,reboundsDefensive_roll10_diff,assists_roll10_diff,turnovers_roll10_diff,steals_roll10_diff,blocks_roll10_diff,plusMinusPoints_roll10_diff,win_roll10_diff
0,11000087,2010-10-18,2010,1610612737,Hawks,0,93.000000,0.469750,0.384500,0.595500,...,-0.019100,-0.157700,0.100000,-7.950000,0.050000,2.55,-0.350000,-4.200000,-28.700000,-0.750000
1,11000110,2010-10-21,2010,1610612737,Hawks,1,89.000000,0.437400,0.345200,0.628200,...,0.005200,-0.167000,-3.000000,-1.800000,1.200000,-1.60,1.200000,-2.200000,-6.600000,-0.200000
2,21000031,2010-10-30,2010,1610612737,Hawks,1,92.444444,0.441556,0.367667,0.695889,...,0.026417,0.027264,-2.055556,1.638889,-1.194444,-0.25,-3.569444,-2.055556,-2.888889,0.069444
3,21000054,2010-11-03,2010,1610612737,Hawks,1,92.000000,0.431500,0.364400,0.721600,...,-0.002600,-0.023200,-0.200000,1.500000,0.100000,-0.70,-0.200000,0.800000,1.100000,0.400000
4,21000090,2010-11-07,2010,1610612737,Hawks,0,95.000000,0.444100,0.304200,0.775800,...,-0.032600,0.040800,-4.600000,2.800000,0.700000,-3.70,-2.000000,0.100000,8.300000,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21735,22501085,2026-03-29,2025,1610612766,Hornets,0,117.300000,0.463500,0.398300,0.796900,...,0.048600,-0.081100,1.200000,-2.500000,1.400000,1.70,1.000000,1.200000,5.600000,0.000000
21736,22501114,2026-04-02,2025,1610612766,Hornets,1,118.700000,0.473400,0.386900,0.817100,...,0.025800,0.090200,-0.400000,3.300000,-0.800000,2.30,-2.200000,-1.200000,9.800000,0.400000
21737,22501120,2026-04-03,2025,1610612766,Hornets,1,119.700000,0.479400,0.376100,0.809700,...,-0.054000,-0.005000,5.100000,2.600000,-8.700000,-0.20,-0.100000,-0.700000,18.500000,0.400000
21738,22501171,2026-04-10,2025,1610612766,Hornets,0,118.200000,0.476200,0.378400,0.807700,...,-0.039800,0.028100,2.100000,1.500000,-5.800000,-2.50,-3.500000,-3.000000,4.400000,0.000000


In [28]:
home_win_pct = []

for year in range(2016, 2026):
    
    home_win_pct.append({
        'Season': year, 
        'Home Win': matchups[matchups['season'] == year]['win'].mean()})
    
home_win_pct_df = pd.DataFrame(home_win_pct)

home_win_pct_df.round(4)

,Season,Home Win
0,2016,0.5800
1,2017,0.5840
2,2018,0.5872
3,2019,0.5372
4,2020,0.5437
5,2021,0.5525
6,2022,0.5744
7,2023,0.5492
8,2024,0.5484
9,2025,0.5523


In [27]:
matchups['win'].mean()

np.float64(0.5735970561177552)

In [ ]:
# if we blindly predict that home team wins for every game, then we have about 57.36% accuracy

In [5]:
matchups['win_roll10_diff'].describe()

count    21740.000000
mean        -0.004244
std          0.290057
min         -1.000000
25%         -0.200000
50%          0.000000
75%          0.200000
max          1.000000
Name: win_roll10_diff, dtype: float64

In [6]:
matchups['win_roll10_diff'].value_counts()

win_roll10_diff
 0.000000    3004
-0.100000    2055
 0.100000    1988
-0.300000    1352
 0.300000    1319
             ... 
 0.188889       1
 0.133333       1
 0.416667       1
-0.571429       1
-0.238095       1
Name: count, Length: 94, dtype: int64

In [7]:
matchups['win_roll10_diff'].abs().value_counts()

win_roll10_diff
0.100000    4043
0.000000    3004
0.300000    2671
0.200000    2423
0.200000    2239
            ... 
0.430556       1
0.714286       1
0.188889       1
0.133333       1
0.238095       1
Name: count, Length: 64, dtype: int64

In [10]:
matchups['pred_win'] = (matchups['win_roll10_diff'] >= 0) * 1.0

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

In [26]:
print(f"Accuracy: {accuracy_score(matchups['win'], matchups['pred_win']):.4f}")

Accuracy: 0.6144


In [29]:
results = []

for year in range(2016, 2026):
    y_true = matchups[matchups['season'] == year]['win']
    y_pred = matchups[matchups['season'] == year]['pred_win']
    results.append({
        'Season': year, 
        'Accuracy': accuracy_score(y_true=y_true, y_pred=y_pred)})
    
results_df = pd.DataFrame(results)

results_df.round(4)

,Season,Accuracy
0,2016,0.6070
1,2017,0.6244
2,2018,0.6025
3,2019,0.6125
4,2020,0.5804
5,2021,0.6108
6,2022,0.5643
7,2023,0.6151
8,2024,0.6257
9,2025,0.6381
